In [25]:
from sentinelhub import SentinelHubCatalog, BBox, CRS, SHConfig
from datetime import date
import requests
import configparser
from utils import get_access_token

In [26]:
config_file = configparser.ConfigParser()
config_file.read("config.ini")

username = config_file["copernicus"]["username"]
password = config_file["copernicus"]["password"]

config = SHConfig()
config.sh_client_id = config_file["copernicus"]["client_id"] #"<CLIENT ID>"
config.sh_client_secret = config_file["copernicus"]["client_secret"] #<CLIENT SECRET>"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token" # Is it required?
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.save("cdse")
config = SHConfig("cdse")

In [27]:

access_token = get_access_token(username, password)

In [28]:

# ID del producto (obtenido con SentinelHubCatalog)
product_id = "S2B_MSIL1C_20220704T104629_N0510_R051_T30SXG_20240630T162125.SAFE"

# URL de descarga
download_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products('{product_id}')/$value"

headers = {
    "Authorization": f"Bearer {access_token}"
}

with requests.get(download_url, headers=headers, stream=True) as r:
    if r.status_code == 200:
        with open(f"{product_id}.zip", 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("✅ Descargado correctamente")
    else:
        print(f"❌ Error {r.status_code}: {r.text}")


❌ Error 422: {"trace-id":"99a4da95b0fa416e872fc0e383475371","code":"DAT-ZIP-111","message":"Invalid uuid as ID parameter"}


In [29]:
def get_product_uuid(product_name, access_token):
    """Busca el UUID real de un producto .SAFE por nombre"""
    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products?$filter=Name eq '{product_name}'&$format=json"
    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        print(f"❌ Error {r.status_code}: {r.text}")
        return None

    results = r.json()
    if not results['value']:
        print("⚠️ Producto no encontrado")
        return None

    return results['value'][0]['Id']  # Este es el UUID correcto

def download_safe_by_uuid(uuid, access_token, output_path):
    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products('{uuid}')/$value"
    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    with requests.get(url, headers=headers, stream=True) as r:
        if r.status_code == 200:
            with open(output_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✅ Descargado: {output_path}")
        else:
            print(f"❌ Error {r.status_code}: {r.text}")

product_name = "S2B_MSIL1C_20220704T104629_N0510_R051_T30SXG.SAFE"

uuid = get_product_uuid(product_name, access_token)
if uuid:
    download_safe_by_uuid(uuid, access_token, f"{product_name}.zip")



❌ Error 404: <html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</center>
</body>
</html>



In [24]:

catalog = SentinelHubCatalog(config=config)

bbox = BBox(bbox=[-0.866977, 37.628916, -0.71696, 37.822802], crs=CRS.WGS84)

search_iterator = catalog.search(
    collection='sentinel-2-l1c',
    bbox=bbox,
    time=('2022-07-01', '2022-07-15'),
    filter='eo:cloud_cover < 70',
    fields={'include': ['id', 'properties.datetime', 'properties.eo:cloud_cover'], 'exclude': []}
)

for result in search_iterator:
    print(result)
    #print(result['id'], result['properties']['datetime'], result['properties']['eo:cloud_cover'])


{'id': 'S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911.SAFE', 'properties': {'datetime': '2022-07-14T11:00:51.599Z', 'eo:cloud_cover': 0.0}}
{'id': 'S2B_MSIL1C_20220714T104629_N0510_R051_T30SYG_20240723T192911.SAFE', 'properties': {'datetime': '2022-07-14T11:00:48.137Z', 'eo:cloud_cover': 0.0}}
{'id': 'S2A_MSIL1C_20220709T105041_N0510_R051_T30SXG_20240703T164651.SAFE', 'properties': {'datetime': '2022-07-09T11:01:00.258Z', 'eo:cloud_cover': 0.07}}
{'id': 'S2A_MSIL1C_20220709T105041_N0510_R051_T30SYG_20240703T164651.SAFE', 'properties': {'datetime': '2022-07-09T11:00:56.782Z', 'eo:cloud_cover': 0.08}}
{'id': 'S2B_MSIL1C_20220704T104629_N0510_R051_T30SXG_20240630T162125.SAFE', 'properties': {'datetime': '2022-07-04T11:00:52.388Z', 'eo:cloud_cover': 14.21}}
{'id': 'S2B_MSIL1C_20220704T104629_N0510_R051_T30SYG_20240630T162125.SAFE', 'properties': {'datetime': '2022-07-04T11:00:48.922Z', 'eo:cloud_cover': 1.92}}


In [23]:
from sentinelhub import SentinelHubCatalog, BBox, CRS
import requests
from requests.auth import HTTPBasicAuth
import os

# Tus credenciales de Copernicus Data Space
USERNAME = username
PASSWORD = password

# Área de interés
bbox = BBox(bbox=[-0.866977, 37.628916, -0.71696, 37.822802], crs=CRS.WGS84)

# Crear carpeta de salida
output_dir = './SAFE_downloads/'
os.makedirs(output_dir, exist_ok=True)

# Conectarse al catálogo
catalog = SentinelHubCatalog(config=config)

# Buscar productos
search_iterator = catalog.search(
    collection='sentinel-2-l1c',
    bbox=bbox,
    time=('2022-07-01', '2022-07-15'),
    filter='eo:cloud_cover < 20',
    fields={'include': ['id', 'properties.datetime', 'properties.eo:cloud_cover'], 'exclude': []}
)

# Descargar por ID
for product in search_iterator:
    product_id = product['id']
    date = product['properties']['datetime']
    cloud = product['properties']['eo:cloud_cover']

    print(f'\n ID: {product_id} | Fecha: {date} | Nubes: {cloud}%')

    # Descargar
    download_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products('{product_id}')/$value"
    output_path = os.path.join(output_dir, f'{product_id}.zip')

    if os.path.exists(output_path):
        print(f'📦 Ya descargado: {output_path}')
        continue

    print(f'⬇️ Descargando a: {output_path}')
    with requests.get(download_url, auth=HTTPBasicAuth(USERNAME, PASSWORD), stream=True) as r:
        if r.status_code == 200:
            with open(output_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f'Descargado: {output_path}')
        else:
            print(f'Error al descargar {product_id} (código {r.status_code})')



 ID: S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911.SAFE | Fecha: 2022-07-14T11:00:51.599Z | Nubes: 0.0%
⬇️ Descargando a: ./SAFE_downloads/S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911.SAFE.zip
Error al descargar S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911.SAFE (código 401)

 ID: S2B_MSIL1C_20220714T104629_N0510_R051_T30SYG_20240723T192911.SAFE | Fecha: 2022-07-14T11:00:48.137Z | Nubes: 0.0%
⬇️ Descargando a: ./SAFE_downloads/S2B_MSIL1C_20220714T104629_N0510_R051_T30SYG_20240723T192911.SAFE.zip
Error al descargar S2B_MSIL1C_20220714T104629_N0510_R051_T30SYG_20240723T192911.SAFE (código 401)

 ID: S2A_MSIL1C_20220709T105041_N0510_R051_T30SXG_20240703T164651.SAFE | Fecha: 2022-07-09T11:01:00.258Z | Nubes: 0.07%
⬇️ Descargando a: ./SAFE_downloads/S2A_MSIL1C_20220709T105041_N0510_R051_T30SXG_20240703T164651.SAFE.zip
Error al descargar S2A_MSIL1C_20220709T105041_N0510_R051_T30SXG_20240703T164651.SAFE (código 401)

 ID: S2A_MSIL1C_20220709T1050